# Football Forecasting — Bonus Dashboard Launcher (Colab)

Use this notebook to start the **FastAPI prediction service** and **Streamlit live replay dashboard** for a presentation.

Recommended use:
1. Open this notebook in Google Colab.
2. Run the cells from top to bottom.
3. Open the dashboard link printed by the **Launch dashboard** cell.
4. Keep the Colab runtime connected while presenting.

The notebook does **not** automatically `git pull`, so it will not overwrite local changes immediately before a presentation.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys
import time
import os

REPO_DIR = Path("/content/drive/MyDrive/ML-Project-Football-Forecasting")
BONUS_DIR = REPO_DIR / "code/bonus_service"
API_PORT = 8000
DASHBOARD_PORT = 8501

if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"Repository not found at {REPO_DIR}. "
        "Update REPO_DIR in this cell if your Drive path is different."
    )

print("Repository:", REPO_DIR)
print("Bonus service:", BONUS_DIR)


## 1. Verify the presentation branch

This checks the repository state without modifying it.


In [ ]:
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=REPO_DIR,
    text=True,
).strip()

status = subprocess.check_output(
    ["git", "status", "--short"],
    cwd=REPO_DIR,
    text=True,
).strip()

latest = subprocess.check_output(
    ["git", "log", "-1", "--oneline"],
    cwd=REPO_DIR,
    text=True,
).strip()

print("Branch:", branch)
print("Latest commit:", latest)
print("Working tree:", "clean" if not status else "has local changes")

if branch != "bonus-service":
    print("\nWARNING: Expected branch 'bonus-service'.")
if status:
    print("\nLocal changes:")
    print(status)


## 2. Install the pinned environment

Leave `INSTALL_DEPENDENCIES = True` for a fresh Colab runtime.  
If the same runtime is already prepared and you need a faster restart, set it to `False`.


In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(REPO_DIR / "code/modeling/requirements_colab_ngboost.txt"),
        "-r", str(BONUS_DIR / "requirements_bonus.txt"),
    ])
    print("Dependencies installed.")
else:
    print("Dependency installation skipped.")


## 3. Start FastAPI

The first API startup loads the trained models and frozen feature tables, so it can take several seconds.


In [ ]:
import requests

def wait_for_http(url, timeout=120, label="service"):
    start = time.time()
    last_error = None
    while time.time() - start < timeout:
        try:
            response = requests.get(url, timeout=3)
            if response.status_code == 200:
                return response
            last_error = f"HTTP {response.status_code}: {response.text[:200]}"
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(2)
    raise RuntimeError(f"{label} did not become ready. Last error: {last_error}")

# Stop stale presentation processes from an earlier run.
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
time.sleep(1)

api_log_path = "/tmp/football_forecasting_api.log"
api_log = open(api_log_path, "w")

api_process = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn", "colab_app:app",
        "--host", "0.0.0.0",
        "--port", str(API_PORT),
    ],
    cwd=BONUS_DIR,
    stdout=api_log,
    stderr=subprocess.STDOUT,
)

health_response = wait_for_http(
    f"http://127.0.0.1:{API_PORT}/health",
    timeout=120,
    label="FastAPI",
)

health = health_response.json()
print("FastAPI: READY")
print("PID:", api_process.pid)
print("Health:", health)
print("Log:", api_log_path)


## 4. Quick backend presentation check

This confirms that held-out matches have real club names and that one live prediction is available.


In [ ]:
BASE_URL = f"http://127.0.0.1:{API_PORT}"

matches = requests.get(
    f"{BASE_URL}/matches",
    params={"split": "test"},
    timeout=20,
).json()

if not matches:
    raise RuntimeError("No held-out matches returned by the API.")

example = matches[0]
example_prediction = requests.get(
    f"{BASE_URL}/predict/{example['match_id']}",
    params={"minute": 60},
    timeout=30,
).json()

print("Held-out matches:", len(matches))
print(
    "Example:",
    f"{example['home_team']} vs {example['away_team']}",
    f"({example.get('match_date', '')})"
)
print("60-minute probabilities:", example_prediction["outcome_probabilities"])
print("Expected margin:", round(example_prediction["expected_final_margin"], 3))
print("SHAP rows:", len(example_prediction.get("top_shap", [])))


## 5. Launch the Streamlit dashboard

The proxy-compatible options below prevent the persistent black/loading-skeleton page that can occur behind the Colab reverse proxy.


In [ ]:
streamlit_log_path = "/tmp/football_forecasting_streamlit.log"
streamlit_log = open(streamlit_log_path, "w")

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "dashboard.py",
        "--server.address", "0.0.0.0",
        "--server.port", str(DASHBOARD_PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableWebsocketCompression", "false",
        "--server.fileWatcherType", "none",
        "--browser.gatherUsageStats", "false",
    ],
    cwd=BONUS_DIR,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

streamlit_health = wait_for_http(
    f"http://127.0.0.1:{DASHBOARD_PORT}/_stcore/health",
    timeout=60,
    label="Streamlit",
)

print("Streamlit: READY")
print("PID:", streamlit_process.pid)
print("Health:", streamlit_health.text.strip())
print("Log:", streamlit_log_path)


## 6. Open the dashboard

Keep this Colab runtime running while the dashboard tab is open.


In [ ]:
from google.colab.output import eval_js
from IPython.display import display, HTML

dashboard_url = eval_js(
    f"google.colab.kernel.proxyPort({DASHBOARD_PORT})"
)

print("Dashboard URL:")
print(dashboard_url)

display(HTML(
    f'''
    <p>
      <a href="{dashboard_url}" target="_blank"
         style="font-size:20px;font-weight:600;">
         Open Football Forecasting Live Replay
      </a>
    </p>
    '''
))


## Presentation flow

A concise defence/demo sequence:

1. Select a **held-out match** with real club names.
2. Move the **Replay position** control from pre-match toward FT.
3. Point out how **Home / Draw / Away** probabilities react to goals and match state.
4. Show the changing **expected final goal margin**.
5. Use the goal annotations on the probability chart to connect events to probability shifts.
6. Show **Why the model predicts this outcome** and highlight the leading TreeSHAP contributions.
7. If asked about serving performance, run the optional latency benchmark below.


## Optional: latency benchmark

This runs 200 HTTP requests against a held-out match and prints p50/p95/p99 evidence.


In [ ]:
match_id = int(matches[0]["match_id"])

benchmark = subprocess.run(
    [
        sys.executable,
        str(BONUS_DIR / "benchmark_latency.py"),
        "--match-id", str(match_id),
        "--minute", "60",
        "--requests", "200",
    ],
    cwd=BONUS_DIR,
    capture_output=True,
    text=True,
)

print(benchmark.stdout)
if benchmark.returncode != 0:
    print(benchmark.stderr)


## Troubleshooting logs

Run this only if the API or dashboard does not start.


In [ ]:
print("=== FastAPI log ===")
if Path("/tmp/football_forecasting_api.log").exists():
    print(Path("/tmp/football_forecasting_api.log").read_text(errors="replace")[-5000:])

print("\n=== Streamlit log ===")
if Path("/tmp/football_forecasting_streamlit.log").exists():
    print(Path("/tmp/football_forecasting_streamlit.log").read_text(errors="replace")[-5000:])


## Stop the presentation services

Run this after the presentation or before restarting the launcher cells.


In [ ]:
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
subprocess.run(["pkill", "-f", "streamlit run"], check=False)

print("FastAPI and Streamlit processes stopped.")
